# Platinum/NEPC-positive patients with low testosterone before ADT

Identify endpoint-positive patients with at least one standardized testosterone result at or below a configurable threshold **strictly before first ADT**, then pull the underlying raw lab, medication, diagnosis, and longitudinal rows for review.

The default `LOW_TESTOSTERONE_MAX_NG_DL = 10` is an EDA threshold, not a clinical definition. Raw `NUMERIC_RESULT`, `TEXT_RESULT`, units, and assay identifiers are retained so below-detection substitutions, unit problems, and unrecorded prior ADT can be audited. Exports contain MRNs and must remain in the approved data environment.

## Configuration

In [ ]:
import sys
from pathlib import Path

import polars as pl

def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / 'COMPASS/survival_analysis/compass_pipeline.py').exists():
            return path
    raise FileNotFoundError('Run this notebook from the PROFILE-testing checkout.')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from COMPASS.survival_analysis import compass_pipeline as cp
from COMPASS.data_preprocessing.longitudinal_data_processing import ADT_ANCHOR_MEDS
from data_preprocessing_common.oncdrs_sources import scan_source

ID_COL = 'DFCI_MRN'
LOW_TESTOSTERONE_MAX_NG_DL = 10.0
RAW_TESTOSTERONE_CODES = {'TES', 'TESTT', 'TOTTES', 'TTEST'}
LONGITUDINAL_PATH = cp._PROFILE_OUTPUT_ROOT / 'longitudinal_prediction_data_adt.csv'
RAW_LABS_PATH = cp.PROFILE_SOURCES['LABS']
RAW_MEDICATIONS_PATH = cp.PROFILE_SOURCES['MEDICATIONS']
RAW_DIAGNOSES_PATH = cp.PROFILE_SOURCES['EHR_DIAGNOSES']
EXPORT_DIR = cp._PROFILE_OUTPUT_ROOT / 'survival_analysis/EDA/pre_adt_low_testosterone'
WRITE_EXPORTS = True

for label, path in {
    'longitudinal': LONGITUDINAL_PATH, 'raw labs': RAW_LABS_PATH,
    'raw medications': RAW_MEDICATIONS_PATH, 'raw diagnoses': RAW_DIAGNOSES_PATH,
}.items():
    print(f'{label:<16} {path}  exists={Path(path).exists()}')
print(f'export directory {EXPORT_DIR}')

## Find endpoint-positive patients

Candidate status is based on the standardized longitudinal table. A patient must be `PLATINUM == 1` or `NEPC == 1` and have a testosterone result `<= LOW_TESTOSTERONE_MAX_NG_DL` with `LAB_DATE < TREATMENT_ANCHOR_DATE`. Same-day results are excluded because their ordering relative to the first ADT administration is unknown.

In [ ]:
required = {
    ID_COL, 'TREATMENT_ANCHOR_DATE', 'LAB_DATE', 'LAB_NAME', 'LAB_VALUE',
    'PLATINUM', 'NEPC', 'PLATINUM_DATE', 'NEPC_DATE',
}
schema_names = set(pl.scan_csv(LONGITUDINAL_PATH, n_rows=0).collect_schema().names())
missing = required - schema_names
if missing:
    raise ValueError(f'{LONGITUDINAL_PATH} is missing required columns: {sorted(missing)}')

longitudinal = (
    pl.scan_csv(LONGITUDINAL_PATH)
    .with_columns(
        pl.col(ID_COL).cast(pl.Int64, strict=False),
        pl.col('LAB_VALUE').cast(pl.Float64, strict=False),
        pl.col('PLATINUM').cast(pl.Int8, strict=False).fill_null(0),
        pl.col('NEPC').cast(pl.Int8, strict=False).fill_null(0),
        pl.col('LAB_DATE').cast(pl.Utf8).str.to_date(strict=False).alias('_LAB_DATE'),
        pl.col('TREATMENT_ANCHOR_DATE').cast(pl.Utf8).str.to_date(strict=False).alias('_ADT_START'),
    )
    .collect(engine='streaming')
)

patient_endpoints = (
    longitudinal.group_by(ID_COL).agg(
        pl.col('PLATINUM').max(), pl.col('NEPC').max(),
        pl.col('PLATINUM_DATE').drop_nulls().first(),
        pl.col('NEPC_DATE').drop_nulls().first(),
        pl.col('_ADT_START').drop_nulls().first(),
    )
    .filter((pl.col('PLATINUM') == 1) | (pl.col('NEPC') == 1))
)

pre_adt_low = (
    longitudinal
    .filter(
        (pl.col('LAB_NAME').str.to_lowercase() == 'testosterone')
        & pl.col('LAB_VALUE').is_not_null()
        & pl.col('LAB_VALUE').is_between(0, LOW_TESTOSTERONE_MAX_NG_DL, closed='both')
        & (pl.col('_LAB_DATE') < pl.col('_ADT_START'))
    )
    .join(patient_endpoints.select(ID_COL), on=ID_COL, how='inner')
)

candidate_summary = (
    pre_adt_low.group_by(ID_COL).agg(
        pl.len().alias('N_LOW_PRE_ADT_TESTOSTERONE'),
        pl.col('LAB_VALUE').min().alias('MIN_PRE_ADT_TESTOSTERONE'),
        pl.col('_LAB_DATE').min().alias('FIRST_LOW_TESTOSTERONE_DATE'),
        pl.col('_LAB_DATE').max().alias('LAST_LOW_TESTOSTERONE_DATE'),
    )
    .join(patient_endpoints, on=ID_COL, how='left')
    .with_columns(
        pl.when((pl.col('PLATINUM') == 1) & (pl.col('NEPC') == 1)).then(pl.lit('Both'))
        .when(pl.col('PLATINUM') == 1).then(pl.lit('Platinum only'))
        .otherwise(pl.lit('NEPC only')).alias('ENDPOINT_GROUP'),
        (pl.col('LAST_LOW_TESTOSTERONE_DATE') - pl.col('_ADT_START'))
        .dt.total_days().alias('LAST_LOW_DAYS_FROM_ADT'),
    )
    .sort(['ENDPOINT_GROUP', 'MIN_PRE_ADT_TESTOSTERONE', ID_COL])
)
candidate_mrns = candidate_summary[ID_COL].to_list()
print(f'{len(candidate_mrns):,} candidate patients at <= {LOW_TESTOSTERONE_MAX_NG_DL:g} ng/dL')
print(candidate_summary.group_by('ENDPOINT_GROUP').agg(pl.len().alias('N_PATIENTS')).sort('ENDPOINT_GROUP'))
candidate_summary

## Standardized testosterone and full longitudinal context

In [ ]:
standardized_testosterone = (
    longitudinal
    .filter(pl.col(ID_COL).is_in(candidate_mrns) & (pl.col('LAB_NAME').str.to_lowercase() == 'testosterone'))
    .with_columns(
        (pl.col('_LAB_DATE') - pl.col('_ADT_START')).dt.total_days().alias('DAYS_FROM_ADT'),
        (pl.col('_LAB_DATE') < pl.col('_ADT_START')).alias('IS_PRE_ADT'),
        ((pl.col('_LAB_DATE') < pl.col('_ADT_START')) & pl.col('LAB_VALUE').is_between(0, LOW_TESTOSTERONE_MAX_NG_DL, closed='both'))
        .alias('MEETS_REVIEW_THRESHOLD'),
    )
    .sort([ID_COL, '_LAB_DATE'])
)
candidate_longitudinal = longitudinal.filter(pl.col(ID_COL).is_in(candidate_mrns)).sort([ID_COL, '_LAB_DATE'])
standardized_testosterone

## Raw source rows

The lab pull deliberately matches `TESTOSTER` in either the raw test code or description and retains every original column. Medication and diagnosis pulls retain all rows for each candidate, which is useful for finding ADT documented before the pipeline anchor or outside the preferred-medication set.

In [ ]:
raw_labs = (
    scan_source(RAW_LABS_PATH)
    .filter(
        pl.col(ID_COL).cast(pl.Float64, strict=False).cast(pl.Int64, strict=False).is_in(candidate_mrns)
        & (
            pl.col('TEST_TYPE_CD').fill_null('').str.to_uppercase().str.strip_chars().is_in(RAW_TESTOSTERONE_CODES)
            | pl.col('TEST_TYPE_CD').fill_null('').str.to_uppercase().str.contains('TESTOSTER')
            | pl.col('TEST_TYPE_DESCR').fill_null('').str.to_uppercase().str.contains('TESTOSTER')
        )
    )
    .collect(engine='streaming')
    .with_columns(
        pl.col(ID_COL).cast(pl.Float64, strict=False).cast(pl.Int64, strict=False),
        pl.col('SPECIMEN_COLLECT_DT').str.to_date(strict=False).alias('_RAW_LAB_DATE'),
    )
    .join(candidate_summary.select(ID_COL, '_ADT_START', 'ENDPOINT_GROUP'), on=ID_COL, how='left')
    .with_columns((pl.col('_RAW_LAB_DATE') - pl.col('_ADT_START')).dt.total_days().alias('DAYS_FROM_ADT'))
    .sort([ID_COL, '_RAW_LAB_DATE'])
)

raw_medications = (
    scan_source(RAW_MEDICATIONS_PATH)
    .filter(pl.col(ID_COL).cast(pl.Float64, strict=False).cast(pl.Int64, strict=False).is_in(candidate_mrns))
    .collect(engine='streaming')
    .with_columns(
        pl.col(ID_COL).cast(pl.Float64, strict=False).cast(pl.Int64, strict=False),
        pl.col('MED_START_DT').str.to_date(strict=False).alias('_MED_DATE'),
        pl.col('NCI_PREFERRED_MED_NM').str.to_uppercase().is_in(sorted(ADT_ANCHOR_MEDS)).alias('IS_PIPELINE_ADT_MED'),
    )
    .join(candidate_summary.select(ID_COL, '_ADT_START', 'ENDPOINT_GROUP'), on=ID_COL, how='left')
    .with_columns((pl.col('_MED_DATE') - pl.col('_ADT_START')).dt.total_days().alias('DAYS_FROM_ADT'))
    .sort([ID_COL, '_MED_DATE'])
)

raw_diagnoses = (
    scan_source(RAW_DIAGNOSES_PATH)
    .filter(pl.col(ID_COL).cast(pl.Float64, strict=False).cast(pl.Int64, strict=False).is_in(candidate_mrns))
    .collect(engine='streaming')
    .with_columns(
        pl.col(ID_COL).cast(pl.Float64, strict=False).cast(pl.Int64, strict=False),
        pl.col('START_DT').str.to_date(strict=False).alias('_DIAGNOSIS_DATE'),
    )
    .join(candidate_summary.select(ID_COL, '_ADT_START', 'ENDPOINT_GROUP'), on=ID_COL, how='left')
    .with_columns((pl.col('_DIAGNOSIS_DATE') - pl.col('_ADT_START')).dt.total_days().alias('DAYS_FROM_ADT'))
    .sort([ID_COL, '_DIAGNOSIS_DATE'])
)

print(f'raw testosterone rows: {raw_labs.height:,}')
print(f'raw medication rows  : {raw_medications.height:,}')
print(f'raw diagnosis rows   : {raw_diagnoses.height:,}')
raw_labs

## Focused medication audit

This view puts pre-anchor medications first. Review non-preferred names, free-text products, orchiectomy evidence in diagnoses, and any ADT-class drug dated before `_ADT_START`.

In [ ]:
medication_audit_cols = [
    ID_COL, 'ENDPOINT_GROUP', '_ADT_START', '_MED_DATE', 'DAYS_FROM_ADT',
    'IS_PIPELINE_ADT_MED', 'NCI_PREFERRED_MED_NM',
]
raw_medications.select([c for c in medication_audit_cols if c in raw_medications.columns]).filter(
    pl.col('DAYS_FROM_ADT') <= 0
).sort([ID_COL, 'DAYS_FROM_ADT'])

## Write review extracts

In [ ]:
exports = {
    'candidate_summary.csv': candidate_summary,
    'standardized_testosterone.csv': standardized_testosterone,
    'raw_testosterone_labs.csv': raw_labs,
    'raw_medications.csv': raw_medications,
    'raw_diagnoses.csv': raw_diagnoses,
    'candidate_longitudinal.parquet': candidate_longitudinal,
}
if WRITE_EXPORTS:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    for filename, frame in exports.items():
        path = EXPORT_DIR / filename
        if path.suffix == '.parquet':
            frame.write_parquet(path)
        else:
            frame.write_csv(path)
        print(f'wrote {frame.height:>8,} rows -> {path}')
else:
    print('WRITE_EXPORTS=False; no files written')

## Suggested review order

1. Sort `candidate_summary` by `MIN_PRE_ADT_TESTOSTERONE` and proximity to ADT.
2. Compare standardized values with raw numeric/text results and raw units. Values created from below-detection strings should be visible in `TEXT_RESULT`.
3. Check `raw_medications` for prior ADT under an unexpected preferred name or a date earlier than the selected anchor.
4. Check diagnoses for orchiectomy, hypogonadism, or outside prostate-cancer treatment context.
5. Change the threshold and rerun to distinguish exact zeroes from the broader near-zero group.